
# RQ3 Notebook: Error Analysis for DistilBERT Misclassification of Neutral Reviews

This notebook answers:

**RQ3:** *What linguistic patterns (contrastive conjunctions, negations, and conditional phrases) contribute to DistilBERT's misclassification of neutral (3-star) reviews?*

## What this notebook does
1. Loads the balanced and random review datasets.
2. Recreates the same train/validation split used in RQ1.
3. Loads your fine-tuned DistilBERT model.
4. Gets predictions on the **balanced validation** set and the **random test** set.
5. Focuses on **neutral reviews only** (`sentiment == 1` in your current label mapping).
6. Detects linguistic markers:
   - negation
   - contrastive conjunctions
   - conditional phrases
7. Runs **chi-square tests** and compares misclassification rates:
   - with marker vs without marker
8. Prints interpretable tables and example errors.

## Hypotheses
For each marker family:

- **H0:** Marker presence and misclassification are independent.
- **Ha:** Marker presence and misclassification are associated.

> In plain words: reviews containing the marker are not more error-prone under H0, but they may be under Ha.


In [ ]:

import os
import re
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer



## 1) File paths

Edit these paths if needed.

- `balanced_path`: cleaned balanced review dataset
- `random_path`: cleaned random review dataset
- `model_path`: your saved fine-tuned DistilBERT model folder

Examples for `model_path`:
- local notebook folder: `./distilbert_final`
- Colab local folder: `/content/distilbert_final`
- Google Drive folder: `/content/drive/MyDrive/distilbert_final`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/DSBA 6165/"

 cleaned_amazon_balanced_reviews.csv
 cleaned_amazon_random_reviews.csv
 distilbert_final_v2.zip
'Lab12_GAN_SV(KhineNwe).ipynb'
 RQ3_DistilBERT_Error_Analysis_ChiSquare.ipynb


In [ ]:
# Load distilbert_final_v1.zip to ./content

# Unzip
# !unzip "/content/drive/MyDrive/DSBA 6165 AI and DL/Project/distilbert_final_v1.zip" -d /content/
!unzip "/content/drive/MyDrive/DSBA 6165/distilbert_final_v2.zip" -d /content/


Archive:  /content/drive/MyDrive/DSBA 6165/distilbert_final_v2.zip
  inflating: /content/tokenizer.json  
  inflating: /content/tokenizer_config.json  
  inflating: /content/config.json    
  inflating: /content/model.safetensors  
  inflating: /content/training_args.bin  


In [ ]:
# Create folder and move these extracted files
import os
import shutil

os.makedirs("/content/distilbert_final_v2", exist_ok=True)

shutil.move("/content/config.json", "/content/distilbert_final_v2/")
shutil.move("/content/model.safetensors", "/content/distilbert_final_v2/")
shutil.move("/content/tokenizer.json", "/content/distilbert_final_v2/")
shutil.move("/content/tokenizer_config.json", "/content/distilbert_final_v2/")

'/content/distilbert_final_v2/tokenizer_config.json'

In [ ]:
# Move balance and random dataset
import os
import shutil

shutil.move("/content/drive/MyDrive/DSBA 6165/cleaned_amazon_balanced_reviews.csv", "/content/sample_data/")

shutil.move("/content/drive/MyDrive/DSBA 6165/cleaned_amazon_random_reviews.csv", "/content/sample_data/")

'/content/sample_data/cleaned_amazon_random_reviews.csv'

# 1. Load DistilBERT model

In [ ]:
# Load Distilbert model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/distilbert_final_v2"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
!ls "/content/distilbert_final_v2"

config.json  model.safetensors	tokenizer_config.json  tokenizer.json


In [ ]:
# Verify the files under sample_data folder
import os
print(os.listdir("/content/sample_data"))

['anscombe.json', 'README.md', 'cleaned_amazon_random_reviews.csv', 'cleaned_amazon_balanced_reviews.csv', 'california_housing_train.csv', 'mnist_train_small.csv', 'mnist_test.csv', 'california_housing_test.csv']


In [ ]:

balanced_path = "/content/sample_data/cleaned_amazon_balanced_reviews.csv"
random_path = "/content/sample_data/cleaned_amazon_random_reviews.csv"

# Change this if your saved model is elsewhere.
model_path = "./distilbert_final_v2"

# Fallback base model name for tokenizer if needed
base_model_name = "distilbert-base-uncased"


In [ ]:

# Basic path checks
print("Balanced exists:", os.path.exists(balanced_path), balanced_path)
print("Random exists:", os.path.exists(random_path), random_path)
print("Model exists:", os.path.exists(model_path), model_path)


Balanced exists: True /content/sample_data/cleaned_amazon_balanced_reviews.csv
Random exists: True /content/sample_data/cleaned_amazon_random_reviews.csv
Model exists: True ./distilbert_final_v2



## 2) Load and clean data

Expected columns:
- `clean_text_final`
- `sentiment`

Current label mapping assumed from your RQ1 notebook:
- `0` = negative
- `1` = neutral (3-star)
- `2` = positive


In [ ]:
# Import Libraries
# Core
import os
import re
import numpy as np
import pandas as pd

# PyTorch
import torch

# Hugging Face
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# Datasets
from datasets import Dataset

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Statistics
from scipy.stats import chi2_contingency

In [ ]:

df_bal = pd.read_csv(balanced_path)
df_rand = pd.read_csv(random_path)

df_bal = df_bal.dropna(subset=["clean_text_final", "sentiment"]).copy()
df_rand = df_rand.dropna(subset=["clean_text_final", "sentiment"]).copy()

df_bal["clean_text_final"] = df_bal["clean_text_final"].astype(str)
df_rand["clean_text_final"] = df_rand["clean_text_final"].astype(str)

df_bal["sentiment"] = df_bal["sentiment"].astype(int)
df_rand["sentiment"] = df_rand["sentiment"].astype(int)

print("Balanced shape:", df_bal.shape)
print("Random shape:", df_rand.shape)
print("\nBalanced sentiment counts:\n", df_bal["sentiment"].value_counts().sort_index())
print("\nRandom sentiment counts:\n", df_rand["sentiment"].value_counts().sort_index())


Balanced shape: (49688, 15)
Random shape: (46352, 16)

Balanced sentiment counts:
 sentiment
0    14946
1    14864
2    19878
Name: count, dtype: int64

Random sentiment counts:
 sentiment
0     6868
1     3235
2    36249
Name: count, dtype: int64



## 3) Recreate the same train/validation split from RQ1
- val_df comes from the balanced dataset split
- test_df comes from the random datase

To compare model behavior on:

a cleaner/balanced validation set
a more natural/random test set

That helps answer whether the model’s errors are consistent across datasets

In [ ]:

X = df_bal["clean_text_final"]
y = df_bal["sentiment"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

val_df = pd.DataFrame({
    "text": X_val.tolist(),
    "label": y_val.tolist()
}).reset_index(drop=True)

test_df = pd.DataFrame({
    "text": df_rand["clean_text_final"].tolist(),
    "label": df_rand["sentiment"].tolist()
}).reset_index(drop=True)

print("Validation size:", len(val_df))
print("Random test size:", len(test_df))


Validation size: 9938
Random test size: 46352



## 4) Load tokenizer and fine-tuned DistilBERT model


In [ ]:

tokenizer_source = model_path if os.path.exists(model_path) else base_model_name
model_source = model_path if os.path.exists(model_path) else base_model_name

tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
model = AutoModelForSequenceClassification.from_pretrained(model_source)

print("Tokenizer loaded from:", tokenizer_source)
print("Model loaded from:", model_source)
print("num_labels:", model.config.num_labels)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokenizer loaded from: ./distilbert_final_v2
Model loaded from: ./distilbert_final_v2
num_labels: 3



## 5) Tokenize datasets and get predictions
Model only needs specific features for prediction


In [ ]:

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

val_ds = Dataset.from_pandas(val_df.copy())
test_ds = Dataset.from_pandas(test_df.copy())


val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# Transformers cannot use raw strings directly
# Need token IDs and attention mask
keep_cols = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in val_ds.column_names:
    keep_cols.append("token_type_ids")

# Removes unnecessary columns like raw text or pandas index columns
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])

trainer = Trainer(model=model)

val_outputs = trainer.predict(val_ds)
val_preds = np.argmax(val_outputs.predictions, axis=1)
val_labels = val_df["label"].to_numpy()

test_outputs = trainer.predict(test_ds)
test_preds = np.argmax(test_outputs.predictions, axis=1)
test_labels = test_df["label"].to_numpy()

print("Balanced validation accuracy:", round(accuracy_score(val_labels, val_preds), 4))
print("Random test accuracy:", round(accuracy_score(test_labels, test_preds), 4))


Map:   0%|          | 0/9938 [00:00<?, ? examples/s]

Map:   0%|          | 0/46352 [00:00<?, ? examples/s]

Balanced validation accuracy: 0.696
Random test accuracy: 0.7556


In [ ]:
# DistilBERT predict on Balance Dataset
np.unique(val_preds, return_counts=True)

(array([0, 1, 2]), array([2919, 3068, 3951]))

Explanation: The distribution of predicted labels confirms that the fine-tuned DistilBERT model produces predictions for all three sentiment classes

- Class 0 (negative) -> 2919
- Class 1 (neutral) -> 3068
- Class 2 (positive) -> 3951

In [ ]:
# DistilBERT predict on Random Dataset
np.unique(test_preds, return_counts=True)

(array([0, 1, 2]), array([ 7652,  9345, 29355]))

**Explanation:**
The distribution of predicted labels confirms that the fine-tuned DistilBERT model produces predictions for all three sentiment classes


*   Class 0 (negative) -> 7652
*   Class 1 (neutral)  ->  9345
*   Class 2 (positive) -> 29255




### DistilBERT Model Classification Summary

In [ ]:
# DistilBERT Model Classification Summary

print("=== DistilBERT | Balanced Validation ===")
print(classification_report(val_labels, val_preds, digits=4))

print("=== DistilBERT | Random Test ===")
print(classification_report(test_labels, test_preds, digits=4))


=== DistilBERT | Balanced Validation ===
              precision    recall  f1-score   support

           0     0.6965    0.6802    0.6882      2989
           1     0.5531    0.5708    0.5618      2973
           2     0.8066    0.8016    0.8041      3976

    accuracy                         0.6960      9938
   macro avg     0.6854    0.6842    0.6847      9938
weighted avg     0.6977    0.6960    0.6968      9938

=== DistilBERT | Random Test ===
              precision    recall  f1-score   support

           0     0.6401    0.7132    0.6747      6868
           1     0.1878    0.5425    0.2790      3235
           2     0.9665    0.7827    0.8649     36249

    accuracy                         0.7556     46352
   macro avg     0.5981    0.6794    0.6062     46352
weighted avg     0.8638    0.7556    0.7958     46352




## 6) Build error-analysis tables

We focus on **neutral reviews only**.

- Neutral label = `1`
- Correct = predicted `1`
- Misclassified = predicted anything other than `1`


In [ ]:
# Only neutral label
NEUTRAL_LABEL = 1

# copy validation data
val_results = val_df.copy()

# Add "pred" column
val_results["pred"] = val_preds

# Mark correct prediction or not, correct prediction => 1 | wron prediction => 0
val_results["correct"] = (val_results["label"] == val_results["pred"]).astype(int)

# Mark misclassification prediction
val_results["misclassified"] = (val_results["label"] != val_results["pred"]).astype(int)

# Add "dataset" type column
val_results["dataset"] = "balanced_validation"


# copy test (random dataset) data
test_results = test_df.copy()

# Add "pred" column
test_results["pred"] = test_preds

test_results["correct"] = (test_results["label"] == test_results["pred"]).astype(int)
test_results["misclassified"] = (test_results["label"] != test_results["pred"]).astype(int)
test_results["dataset"] = "random_test"

# Filter only neutral reviews
neutral_val = val_results[val_results["label"] == NEUTRAL_LABEL].copy()
neutral_test = test_results[test_results["label"] == NEUTRAL_LABEL].copy()

print("Neutral reviews in balanced validation:", len(neutral_val))
print("Neutral reviews in random test:", len(neutral_test))

print("\nBalanced neutral misclassification rate:",
      round(neutral_val["misclassified"].mean(), 4))
print("Random-test neutral misclassification rate:",
      round(neutral_test["misclassified"].mean(), 4))


Neutral reviews in balanced validation: 2973
Neutral reviews in random test: 3235

Balanced neutral misclassification rate: 0.4292
Random-test neutral misclassification rate: 0.4575


**Explanation of neutral misclassification**


*  Neutral sentiment is harder to classify because it often contains **linguistic markers** that change or obscure meaning
*   **Negation** (e.g., “not”) can reverse sentiment polarity, making it difficult for the model to interpret the correct meaning


*   **Contrast words** (e.g., “but”, “however”) introduce both positive and negative signals in the same sentence, creating conflicting information.
*   **Conditional expressions** (e.g., “would”, “if”) make sentiment hypothetical and less definite, weakening clear sentiment signals


*   These patterns lead to **ambiguity and mixed sentiment**, which are harder for the model to learn and predict accurately
*   As a result, neutral reviews have **higher misclassification rates** compared to other classes



*   This issue becomes more pronounced in **real-world (random) datasets**, where such complex language patterns appear more frequently










In [ ]:
val_results.groupby("label")["misclassified"].mean()

,misclassified
label,
0,0.319839
1,0.429196
2,0.198441


In [ ]:

test_results.groupby("label")["misclassified"].mean()

,misclassified
label,
0,0.286838
1,0.457496
2,0.217330



## 7) Define linguistic markers

You can expand these lists, but keep them transparent and explainable in your report.


In [ ]:

NEGATION_PATTERNS = [
    r"\bnot\b", r"\bno\b", r"\bnever\b", r"\bnone\b", r"\bnothing\b",
    r"\bneither\b", r"\bnor\b", r"\bwithout\b", r"\bdon't\b", r"\bdoesn't\b",
    r"\bdidn't\b", r"\bwon't\b", r"\bwouldn't\b", r"\bcouldn't\b", r"\bcan't\b",
    r"\bisn't\b", r"\baren't\b", r"\bwasn't\b", r"\bweren't\b", r"\bhasn't\b",
    r"\bhaven't\b", r"\bhadn't\b"
]

CONTRAST_PATTERNS = [
    r"\bbut\b", r"\bhowever\b", r"\bthough\b", r"\balthough\b",
    r"\byet\b", r"\bstill\b", r"\beven though\b", r"\bon the other hand\b",
    r"\bwhile\b", r"\bwhereas\b"
]

CONDITIONAL_PATTERNS = [
    r"\bif\b", r"\bif only\b", r"\bunless\b", r"\bprovided that\b",
    r"\bassuming\b", r"\bwould\b", r"\bcould\b", r"\bshould\b",
    r"\bmight\b", r"\bmay\b"
]

def contains_any_pattern(text, patterns):
    text = str(text).lower()
    return int(any(re.search(p, text) for p in patterns))

def add_marker_columns(df):
    out = df.copy()
    out["has_negation"] = out["text"].apply(lambda x: contains_any_pattern(x, NEGATION_PATTERNS))
    out["has_contrast"] = out["text"].apply(lambda x: contains_any_pattern(x, CONTRAST_PATTERNS))
    out["has_conditional"] = out["text"].apply(lambda x: contains_any_pattern(x, CONDITIONAL_PATTERNS))
    out["has_any_marker"] = (
        (out["has_negation"] == 1) |
        (out["has_contrast"] == 1) |
        (out["has_conditional"] == 1)
    ).astype(int)
    return out

neutral_val = add_marker_columns(neutral_val)
neutral_test = add_marker_columns(neutral_test)

neutral_val[["text", "misclassified", "has_negation", "has_contrast", "has_conditional", "has_any_marker"]].head()


,text,misclassified,has_negation,has_contrast,has_conditional,has_any_marker
0,fyi content label sock countri origin mark soc...,1,0,0,0,0
2,bought sale thought would good hold phone firs...,0,0,1,1,1
6,decent skate wheel could better qualiti son le...,0,0,0,1,1
11,product alright find bit wide fit shoulder blade,0,0,0,0,0
12,comfort would hope,0,0,0,1,1


In [ ]:
neutral_test[["text", "misclassified", "has_negation", "has_contrast", "has_conditional", "has_any_marker"]].head()

,text,misclassified,has_negation,has_contrast,has_conditional,has_any_marker
20,work well clear view mask leak underwat snorke...,0,0,0,0,0
79,comfort oem seem stay togeth well either,0,0,0,0,0
81,super warm nice fluffi zipper drive freak insa...,1,0,0,0,0
89,strict averag glove fit alway felt like lack s...,0,0,0,0,0
105,said iron stake put ground came without replac...,1,1,0,0,1



## 8) Chi-square test and misclassification-rate comparison

For each marker family, we create a 2×2 table:

| Marker present? | Misclassified | Correct |
|---|---:|---:|
| Yes | a | b |
| No  | c | d |

Then we run a chi-square test.


In [ ]:

def marker_analysis(df, marker_col, dataset_name):
    contingency = pd.crosstab(df[marker_col], df["misclassified"])

    # Ensure a full 2x2 shape
    contingency = contingency.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    contingency.index = ["without_marker", "with_marker"]
    contingency.columns = ["correct", "misclassified"]

    chi2, p, dof, expected = chi2_contingency(contingency)

    with_rate = contingency.loc["with_marker", "misclassified"] / contingency.loc["with_marker"].sum() if contingency.loc["with_marker"].sum() > 0 else np.nan
    without_rate = contingency.loc["without_marker", "misclassified"] / contingency.loc["without_marker"].sum() if contingency.loc["without_marker"].sum() > 0 else np.nan

    result = pd.DataFrame({
        "dataset": [dataset_name],
        "marker": [marker_col],
        "with_marker_n": [int(contingency.loc["with_marker"].sum())],
        "without_marker_n": [int(contingency.loc["without_marker"].sum())],
        "misclass_rate_with_marker": [with_rate],
        "misclass_rate_without_marker": [without_rate],
        "rate_difference": [with_rate - without_rate if pd.notna(with_rate) and pd.notna(without_rate) else np.nan],
        "chi2": [chi2],
        "dof": [dof],
        "p_value": [p]
    })
    return contingency, pd.DataFrame(expected, index=contingency.index, columns=contingency.columns), result

def run_all_marker_tests(df, dataset_name):
    summary_rows = []
    outputs = {}
    for marker in ["has_negation", "has_contrast", "has_conditional", "has_any_marker"]:
        contingency, expected, result = marker_analysis(df, marker, dataset_name)
        outputs[marker] = {
            "contingency": contingency,
            "expected": expected,
            "summary": result
        }
        summary_rows.append(result)
    summary_df = pd.concat(summary_rows, ignore_index=True)
    return outputs, summary_df

val_outputs_markers, val_summary = run_all_marker_tests(neutral_val, "balanced_validation")
test_outputs_markers, test_summary = run_all_marker_tests(neutral_test, "random_test")


In [ ]:

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("=== Balanced Validation: Neutral Review Marker Analysis ===")
display(val_summary)

print("=== Random Test: Neutral Review Marker Analysis ===")
display(test_summary)


=== Balanced Validation: Neutral Review Marker Analysis ===


,dataset,marker,with_marker_n,without_marker_n,misclass_rate_with_marker,misclass_rate_without_marker,rate_difference,chi2,dof,p_value
0,balanced_validation,has_negation,197,2776,0.4822,0.4254,0.0568,2.1962,1,0.1384
1,balanced_validation,has_contrast,476,2497,0.3361,0.4469,-0.1108,19.5849,1,0.0000
2,balanced_validation,has_conditional,905,2068,0.3713,0.4545,-0.0833,17.4809,1,0.0000
3,balanced_validation,has_any_marker,1240,1733,0.3726,0.4697,-0.0971,27.4370,1,0.0000


=== Random Test: Neutral Review Marker Analysis ===


,dataset,marker,with_marker_n,without_marker_n,misclass_rate_with_marker,misclass_rate_without_marker,rate_difference,chi2,dof,p_value
0,random_test,has_negation,157,3078,0.5541,0.4526,0.1016,5.8071,1,0.0160
1,random_test,has_contrast,380,2855,0.3158,0.4764,-0.1606,34.1932,1,0.0000
2,random_test,has_conditional,824,2411,0.4041,0.4757,-0.0716,12.4015,1,0.0004
3,random_test,has_any_marker,1117,2118,0.3930,0.4915,-0.0985,28.1837,1,0.0000


**Summary**
A comparison between the balanced validation and random test datasets shows that negation significantly increases misclassification in the test dataset, while contrast and conditional markers consistently reduce error rates across both datasets. These results suggest that linguistic markers do not uniformly degrade performance; instead, their impact depends on how they influence sentence structure and meaning


## 9) Inspect contingency tables

These are the actual counts used by the chi-square test.


In [ ]:

for marker, out in val_outputs_markers.items():
    print(f"\n=== Balanced Validation | {marker} ===")
    display(out["contingency"])
    print("Expected counts:")
    display(out["expected"])

for marker, out in test_outputs_markers.items():
    print(f"\n=== Random Test | {marker} ===")
    display(out["contingency"])
    print("Expected counts:")
    display(out["expected"])



=== Balanced Validation | has_negation ===


,correct,misclassified
without_marker,1595,1181
with_marker,102,95


Expected counts:


,correct,misclassified
without_marker,1584.5516,1191.4484
with_marker,112.4484,84.5516



=== Balanced Validation | has_contrast ===


...

,correct,misclassified
without_marker,1307.9768,1103.0232
with_marker,447.0232,376.9768



=== Random Test | has_any_marker ===


,correct,misclassified
without_marker,1077,1041
with_marker,678,439


Expected counts:


,correct,misclassified
without_marker,1149.0232,968.9768
with_marker,605.9768,511.0232



## 10) Human-readable interpretation helper

This prints a short interpretation for each test.


In [ ]:

def interpret_results(summary_df, alpha=0.05):
    for _, row in summary_df.iterrows():
        marker = row["marker"]
        dataset = row["dataset"]
        p = row["p_value"]
        r_with = row["misclass_rate_with_marker"]
        r_without = row["misclass_rate_without_marker"]
        diff = row["rate_difference"]

        print(f"Dataset: {dataset} | Marker: {marker}")
        print(f"  Misclassification rate with marker:    {r_with:.4f}")
        print(f"  Misclassification rate without marker: {r_without:.4f}")
        print(f"  Difference: {diff:.4f}")

        if p < alpha:
            print(f"  Result: Reject H0 (p = {p:.4g}). Marker presence is significantly associated with misclassification.")
        else:
            print(f"  Result: Fail to reject H0 (p = {p:.4g}). No statistically significant association detected.")
        print("-" * 80)

print("Balanced validation interpretations")
interpret_results(val_summary)

print("\nRandom test interpretations")
interpret_results(test_summary)


Balanced validation interpretations
Dataset: balanced_validation | Marker: has_negation
  Misclassification rate with marker:    0.4822
  Misclassification rate without marker: 0.4254
  Difference: 0.0568
  Result: Fail to reject H0 (p = 0.1384). No statistically significant association detected.
--------------------------------------------------------------------------------
Dataset: balanced_validation | Marker: has_contrast
  Misclassification rate with marker:    0.3361
  Misclassification rate without marker: 0.4469
  Difference: -0.1108
  Result: Reject H0 (p = 9.622e-06). Marker presence is significantly associated with misclassification.
--------------------------------------------------------------------------------
Dataset: balanced_validation | Marker: has_conditional
  Misclassification rate with marker:    0.3713
  Misclassification rate without marker: 0.4545
  Difference: -0.0833
  Result: Reject H0 (p = 2.902e-05). Marker presence is significantly associated with miscla


## 11) Optional: Look at example neutral reviews that DistilBERT got wrong

This is useful for qualitative error analysis.


In [ ]:

def show_examples(df, marker_col=None, misclassified=1, n=10, random_state=42):
    subset = df[df["misclassified"] == misclassified].copy()
    if marker_col is not None:
        subset = subset[subset[marker_col] == 1].copy()
    cols = ["text", "label", "pred", "misclassified", "has_negation", "has_contrast", "has_conditional", "has_any_marker"]
    if len(subset) == 0:
        print("No matching examples.")
        return
    display(subset[cols].sample(min(n, len(subset)), random_state=random_state))

print("Balanced validation: misclassified neutral reviews with any marker")
show_examples(neutral_val, marker_col="has_any_marker", misclassified=1, n=10)

print("Random test: misclassified neutral reviews with any marker")
show_examples(neutral_test, marker_col="has_any_marker", misclassified=1, n=10)


Balanced validation: misclassified neutral reviews with any marker


,text,label,pred,misclassified,has_negation,has_contrast,has_conditional,has_any_marker
5100,flag made good materi thick hold hard florida ...,1,2,1,0,1,0,1
676,brim pad stiff uncomfort would buy,1,0,1,0,0,1,1
852,want littl kia soul disappoint arriv list say ...,1,0,1,1,0,1,1
4994,definit top qualiti blanket come quit handi pa...,1,2,1,0,0,1,1
2710,like idea block yogi might wrist issu problem ...,1,2,1,0,0,1,1
4622,tri add safeti vest would stick,1,0,1,0,0,1,1
8635,big return actual plus size like said never re...,1,0,1,1,0,0,1
4773,one ounc ideal week long travel realiz small o...,1,0,1,0,0,1,1
9802,order think set dumb bell almost mention fact ...,1,0,1,0,0,1,1
1693,seem like realli good idea bought pictur abl p...,1,0,1,0,0,1,1


Random test: misclassified neutral reviews with any marker


,text,label,pred,misclassified,has_negation,has_contrast,has_conditional,has_any_marker
28645,gift nostalg parent certain fit bill giant ena...,1,0,1,0,1,0,1
8590,descript said would fit wide mouth bottl howev...,1,0,1,0,0,1,1
36483,hold love pocket sure wick abil yet far seem l...,1,2,1,0,1,0,1
26858,stay phone well color price great rubber shock...,1,2,1,0,1,0,1
34623,would assum would good beginn set end like blo...,1,0,1,0,0,1,1
31417,use workout class like fact use carpet hard su...,1,2,1,0,0,1,1
19859,hat deliv brim curl unabl get flat tri put wei...,1,0,1,1,0,1,1
8892,base size price reduct still consid good buy h...,1,0,1,0,1,1,1
45833,fall small athlet curvi thick thigh butt small...,1,0,1,0,0,1,1
42362,ship miss coupl screw need instal understand s...,1,0,1,1,1,0,1



## 12) Optional extension: Compare DistilBERT vs TF-IDF + Logistic Regression on the same RQ3 setup

This is **optional**. Your main RQ3 focuses on DistilBERT only.  
But if you later want, you can run the same marker analysis on the logistic-regression predictions and compare whether BERT is more or less sensitive to these markers.



## 13) Suggested report wording

Use this structure in your write-up:

- **Error definition:** A neutral review was treated as misclassified when DistilBERT predicted a label other than neutral.
- **Markers:** We identified negation, contrastive conjunction, and conditional markers using rule-based lexical patterns.
- **Statistical test:** For each marker family, a 2×2 contingency table was constructed and a chi-square test of independence was applied.
- **H0:** Marker presence is independent of misclassification.
- **Ha:** Marker presence is associated with misclassification.
- **Interpretation:** We compared misclassification rates for reviews with and without markers and used p-values to determine whether the association was statistically significant.
